In [ ]:
!pip install boto3 pymysql


In [ ]:
import boto3
import pymysql
import pandas as pd
import os

# ---------- RDS Configuration ----------
rds_host = ''
rds_user = ''
rds_password = ''  # 🔒 REPLACE
rds_database = 'youtube_db'
rds_table = 'youtube_videos'

# ---------- S3 Configuration ----------
bucket_name = ''
s3_key = 'youtube-data/youtube_videos.csv'
local_path = '/tmp/youtube_videos.csv'

# ---------- Connect to RDS ----------
conn = pymysql.connect(
    host=rds_host,
    user=rds_user,
    password=rds_password,
    database=rds_database
)

# ---------- Fetch data into DataFrame ----------
query = f"SELECT * FROM {rds_table}"
df = pd.read_sql(query, conn)
conn.close()

# ---------- Clean & Export CSV ----------
# Replace newlines, ensure strings, avoid shifting
df = df.fillna('').astype(str)
df = df.applymap(lambda x: x.replace('\n', ' ').replace('\r', ' '))

# Write with quoting
df.to_csv(local_path, index=False, quoting=1)  # 1 = csv.QUOTE_ALL

# ---------- Upload to S3 ----------
s3 = boto3.client(
    's3',
    aws_access_key_id='',          # 🔒 Replace or remove if using EC2/IAM role
    aws_secret_access_key='',
    region_name=''
)

s3.upload_file(local_path, bucket_name, s3_key)
print(f"✅ Uploaded clean DataFrame CSV to s3://{bucket_name}/{s3_key}")


/tmp/ipython-input-8-1560531884.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
/tmp/ipython-input-8-1560531884.py:34: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.replace('\n', ' ').replace('\r', ' '))


✅ Uploaded clean DataFrame CSV to s3://ashishbucket08/youtube-data/youtube_videos.csv
